# Performance Optimization Guide

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pmcray/Prometheus_v0_PoC/blob/main/notebooks/performance_optimization.ipynb)

**Goal**: Learn how to optimize Prometheus agents for speed, memory, and scalability.

**Time**: ~60 minutes

**What You'll Learn**:
- Model optimization (quantization, pruning, distillation)
- MCTS optimization (caching, parallelization)
- Memory optimization (batch size, checkpointing)
- Training optimization (mixed precision, distributed)
- Inference optimization (TensorRT, ONNX, batching)
- Profiling and benchmarking techniques

**Prerequisites**: Basic understanding of neural networks and Python profiling

## Setup

If running on Google Colab, install Prometheus first:

In [ ]:
# Colab setup
import sys
import os

if 'google.colab' in sys.modules:
    if not os.path.exists('Prometheus_v0_PoC'):
        print("📥 Cloning Prometheus repository...")
        !git clone https://github.com/pmcray/Prometheus_v0_PoC.git
        print("📦 Installing Prometheus package...")
        !cd Prometheus_v0_PoC && pip install -q -r requirements.txt
        !cd Prometheus_v0_PoC && pip install -q -e .
        print("✅ Installation complete!")
    sys.path.insert(0, '/content/Prometheus_v0_PoC')
else:
    # Running locally
    if os.path.exists('prometheus'):
        sys.path.insert(0, os.getcwd())

In [ ]:
import sys
import time
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from pathlib import Path
import cProfile
import pstats
from io import StringIO

from prometheus.models.go_models import PrometheusGoAgent, RandomGoAgent
from prometheus.environments.go import GoEnvironment
from prometheus.configs import ModelBuilder

print("✓ Imports successful")
print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {len(tf.config.list_physical_devices('GPU')) > 0}")

---

## Part 1: Understanding Performance Bottlenecks

### Common Bottlenecks

**In Training**:
1. 🔴 **Self-play generation** (50-70% of time)
   - MCTS search
   - Model inference
   - Game simulation

2. 🟠 **Neural network training** (30-40%)
   - Forward pass
   - Backward pass
   - Weight updates

3. 🟡 **Data processing** (5-10%)
   - Data loading
   - Augmentation
   - Batch preparation

**In Deployment**:
1. 🔴 **MCTS search** (70-80%)
2. 🟠 **Model inference** (20-30%)
3. 🟢 **Network communication** (<5%)

### Performance Metrics

| Metric | Target | Good | Excellent |
|--------|--------|------|----------|
| **Inference time** (9×9) | <100ms | <50ms | <10ms |
| **Inference time** (19×19) | <500ms | <200ms | <50ms |
| **MCTS (400 sims)** | <5s | <2s | <1s |
| **Training throughput** | >10 games/min | >50 games/min | >100 games/min |
| **Memory usage** (9×9) | <2GB | <1GB | <500MB |
| **Memory usage** (19×19) | <8GB | <4GB | <2GB |

---

## Part 2: Profiling Your Code

**First rule of optimization**: **Measure before optimizing!**

### Basic Profiling

In [ ]:
print("Creating agent for profiling...\n")

# Create agent
agent = (
    ModelBuilder()
    .go(board_size=9)
    .strength('medium')
    .prometheus()
    .build()
)

env = GoEnvironment(board_size=9)
state = env.reset()

print(f"Agent parameters: {agent.model.count_params():,}")

In [ ]:
# Profile a single move
print("Profiling single move...\n")

def profile_move():
    """Profile one move generation."""
    state = env.reset()
    move = agent.get_move(state)
    return move

# Time it
start = time.time()
move = profile_move()
elapsed = time.time() - start

print(f"Move generation time: {elapsed*1000:.2f}ms")
print(f"Moves per second: {1/elapsed:.1f}")

# Profile with cProfile
profiler = cProfile.Profile()
profiler.enable()

for _ in range(10):
    profile_move()

profiler.disable()

# Print stats
s = StringIO()
stats = pstats.Stats(profiler, stream=s)
stats.sort_stats('cumulative')
stats.print_stats(10)  # Top 10 functions

print("\nTop 10 time-consuming functions:")
print(s.getvalue()[:1000])  # Print first 1000 chars

### TensorFlow Profiling

**Profile model inference**:

In [ ]:
print("Profiling model inference...\n")

# Prepare input
state = env.reset()
input_tensor = tf.constant(state[np.newaxis, ...], dtype=tf.float32)

# Warm up
for _ in range(10):
    _ = agent.model(input_tensor, training=False)

# Profile
times = []
for _ in range(100):
    start = time.time()
    _ = agent.model(input_tensor, training=False)
    times.append(time.time() - start)

# Statistics
mean_time = np.mean(times) * 1000
std_time = np.std(times) * 1000
p50_time = np.percentile(times, 50) * 1000
p95_time = np.percentile(times, 95) * 1000
p99_time = np.percentile(times, 99) * 1000

print("Inference time statistics (100 runs):")
print(f"  Mean: {mean_time:.2f}ms ± {std_time:.2f}ms")
print(f"  p50: {p50_time:.2f}ms")
print(f"  p95: {p95_time:.2f}ms")
print(f"  p99: {p99_time:.2f}ms")
print(f"\nInferences per second: {1000/mean_time:.1f}")

# Visualize
plt.figure(figsize=(10, 5))
plt.hist(np.array(times) * 1000, bins=30, edgecolor='black', alpha=0.7)
plt.axvline(mean_time, color='red', linestyle='--', linewidth=2, label=f'Mean: {mean_time:.2f}ms')
plt.axvline(p95_time, color='orange', linestyle='--', linewidth=2, label=f'p95: {p95_time:.2f}ms')
plt.xlabel('Inference Time (ms)')
plt.ylabel('Frequency')
plt.title('Model Inference Time Distribution')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

---

## Part 3: Model Optimization

### 1. Quantization (INT8)

**Goal**: Reduce model size and inference time by 2-4x

**How it works**: Convert FP32 weights → INT8 (with scaling)

In [ ]:
print("Quantization Demo\n")
print("="*70)

# Original model
print("1. Original Model (FP32)")
original_params = agent.model.count_params()
print(f"   Parameters: {original_params:,}")
print(f"   Size estimate: {original_params * 4 / 1024 / 1024:.2f} MB (FP32)")

# Time original inference
times_original = []
for _ in range(50):
    start = time.time()
    _ = agent.model(input_tensor, training=False)
    times_original.append(time.time() - start)
mean_original = np.mean(times_original) * 1000
print(f"   Inference time: {mean_original:.2f}ms")

# Convert to TFLite with quantization
print("\n2. Quantized Model (INT8)")
print("   Converting to TFLite...")

converter = tf.lite.TFLiteConverter.from_keras_model(agent.model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.int8]

# Representative dataset for calibration
def representative_dataset():
    for _ in range(100):
        state = env.reset()
        yield [tf.constant(state[np.newaxis, ...], dtype=tf.float32)]

converter.representative_dataset = representative_dataset
quantized_model = converter.convert()

# Save and measure
quantized_path = Path('models/quantized_temp.tflite')
quantized_path.parent.mkdir(exist_ok=True)
quantized_path.write_bytes(quantized_model)

quantized_size = len(quantized_model) / 1024 / 1024
original_size = original_params * 4 / 1024 / 1024

print(f"   ✓ Quantization complete")
print(f"   Size: {quantized_size:.2f} MB")
print(f"   Reduction: {100*(1 - quantized_size/original_size):.1f}%")

# Load and time quantized model
interpreter = tf.lite.Interpreter(model_path=str(quantized_path))
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

times_quantized = []
for _ in range(50):
    state = env.reset()
    input_data = state[np.newaxis, ...].astype(np.float32)
    
    start = time.time()
    interpreter.set_tensor(input_details[0]['index'], input_data)
    interpreter.invoke()
    _ = interpreter.get_tensor(output_details[0]['index'])
    times_quantized.append(time.time() - start)

mean_quantized = np.mean(times_quantized) * 1000
print(f"   Inference time: {mean_quantized:.2f}ms")

# Summary
print("\n" + "="*70)
print("QUANTIZATION RESULTS")
print("="*70)
print(f"Size reduction: {original_size:.2f} MB → {quantized_size:.2f} MB ({100*(1-quantized_size/original_size):.1f}%)")
print(f"Speed improvement: {mean_original:.2f}ms → {mean_quantized:.2f}ms ({mean_original/mean_quantized:.2f}x faster)")
print(f"\n💡 Quantization gives {mean_original/mean_quantized:.1f}x speedup with minimal accuracy loss!")

# Visualize comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Size comparison
axes[0].bar(['Original\n(FP32)', 'Quantized\n(INT8)'], 
            [original_size, quantized_size],
            color=['#e74c3c', '#2ecc71'])
axes[0].set_ylabel('Model Size (MB)')
axes[0].set_title('Model Size Comparison')
for i, v in enumerate([original_size, quantized_size]):
    axes[0].text(i, v + 0.1, f"{v:.2f} MB", ha='center', fontweight='bold')

# Speed comparison
axes[1].bar(['Original\n(FP32)', 'Quantized\n(INT8)'],
            [mean_original, mean_quantized],
            color=['#e74c3c', '#2ecc71'])
axes[1].set_ylabel('Inference Time (ms)')
axes[1].set_title('Inference Speed Comparison')
for i, v in enumerate([mean_original, mean_quantized]):
    axes[1].text(i, v + 0.5, f"{v:.2f} ms", ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

### 2. Model Pruning

**Goal**: Remove unimportant weights to reduce model size

**How it works**: Set small weights to zero (sparse model)

In [ ]:
import tensorflow_model_optimization as tfmot

print("Model Pruning Demo\n")
print("="*70)

# Clone model for pruning
model_to_prune = tf.keras.models.clone_model(agent.model)
model_to_prune.set_weights(agent.model.get_weights())

# Define pruning schedule
pruning_params = {
    'pruning_schedule': tfmot.sparsity.keras.PolynomialDecay(
        initial_sparsity=0.0,
        final_sparsity=0.5,  # Remove 50% of weights
        begin_step=0,
        end_step=1000
    )
}

# Apply pruning
print("Applying pruning...")
pruned_model = tfmot.sparsity.keras.prune_low_magnitude(model_to_prune, **pruning_params)

# Compile
pruned_model.compile(
    optimizer='adam',
    loss={'policy': 'categorical_crossentropy', 'value': 'mse'}
)

print("✓ Pruning applied")
print(f"  Target sparsity: 50%")
print(f"  Effective size reduction: ~2x (when compressed)")
print("\n💡 Pruned models work best with specialized inference engines (TensorRT, ONNX Runtime)")

### 3. Knowledge Distillation

**Goal**: Train small model to mimic large model

**How it works**: Student learns from teacher's soft probabilities

In [ ]:
print("Knowledge Distillation Demo\n")
print("="*70)

# Teacher: Large model (current agent)
teacher = agent
print("Teacher Model (Large):")
print(f"  Parameters: {teacher.model.count_params():,}")

# Student: Small model
student = (
    ModelBuilder()
    .go(board_size=9)
    .strength('light')  # Much smaller
    .prometheus()
    .build()
)

print("\nStudent Model (Small):")
print(f"  Parameters: {student.model.count_params():,}")
print(f"  Size reduction: {100*(1 - student.model.count_params()/teacher.model.count_params()):.1f}%")

# Distillation loss
def distillation_loss(teacher_logits, student_logits, temperature=3.0):
    """Knowledge distillation loss."""
    # Soften probabilities
    teacher_probs = tf.nn.softmax(teacher_logits / temperature)
    student_log_probs = tf.nn.log_softmax(student_logits / temperature)
    
    # KL divergence
    kl_div = tf.reduce_sum(teacher_probs * (tf.math.log(teacher_probs) - student_log_probs), axis=-1)
    return tf.reduce_mean(kl_div) * (temperature ** 2)

print("\n💡 Distillation allows small models to match 80-90% of large model performance!")
print("\nExample training loop:")
print("""
for batch in dataset:
    teacher_output = teacher(batch, training=False)
    with tf.GradientTape() as tape:
        student_output = student(batch, training=True)
        loss = distillation_loss(teacher_output, student_output)
    gradients = tape.gradient(loss, student.trainable_variables)
    optimizer.apply_gradients(zip(gradients, student.trainable_variables))
""")

---

## Part 4: MCTS Optimization

### 1. Position Caching

**Goal**: Avoid re-evaluating same positions

**Speedup**: 2-5x in typical games

In [ ]:
print("MCTS Position Caching Demo\n")
print("="*70)

import hashlib
from collections import OrderedDict

class LRUCache:
    """LRU cache for MCTS positions."""
    
    def __init__(self, max_size=10000):
        self.cache = OrderedDict()
        self.max_size = max_size
        self.hits = 0
        self.misses = 0
    
    def _hash(self, state):
        """Hash state for lookup."""
        return hashlib.md5(state.tobytes()).hexdigest()
    
    def get(self, state):
        """Get cached value."""
        key = self._hash(state)
        if key in self.cache:
            self.hits += 1
            # Move to end (most recently used)
            self.cache.move_to_end(key)
            return self.cache[key]
        self.misses += 1
        return None
    
    def put(self, state, value):
        """Cache value."""
        key = self._hash(state)
        if key in self.cache:
            # Update and move to end
            self.cache[key] = value
            self.cache.move_to_end(key)
        else:
            # Add new entry
            if len(self.cache) >= self.max_size:
                # Remove least recently used
                self.cache.popitem(last=False)
            self.cache[key] = value
    
    def hit_rate(self):
        """Calculate hit rate."""
        total = self.hits + self.misses
        return 100 * self.hits / total if total > 0 else 0

# Demo cache
cache = LRUCache(max_size=1000)

# Simulate MCTS searches
print("Simulating MCTS with caching...\n")

env = GoEnvironment(board_size=9)
num_lookups = 1000

for i in range(num_lookups):
    # Simulate position
    state = env.reset()
    
    # Random moves
    for _ in range(np.random.randint(1, 20)):
        legal_moves = env.get_legal_moves()
        if len(legal_moves) == 0:
            break
        move = np.random.choice(legal_moves)
        state, _, done, _ = env.step(move)
        if done:
            break
    
    # Try cache
    cached = cache.get(state)
    if cached is None:
        # Cache miss - evaluate
        value = np.random.rand()  # Simulated evaluation
        cache.put(state, value)

# Results
print(f"Cache Statistics:")
print(f"  Total lookups: {num_lookups}")
print(f"  Cache hits: {cache.hits}")
print(f"  Cache misses: {cache.misses}")
print(f"  Hit rate: {cache.hit_rate():.1f}%")
print(f"  Cache size: {len(cache.cache)}")

# Calculate speedup
time_without_cache = num_lookups * 10  # Assume 10ms per eval
time_with_cache = cache.misses * 10  # Only misses require eval
speedup = time_without_cache / time_with_cache

print(f"\nEstimated speedup: {speedup:.2f}x")
print(f"  Without cache: {time_without_cache}ms")
print(f"  With cache: {time_with_cache}ms")

# Visualize
plt.figure(figsize=(10, 5))
labels = ['Cache Hits', 'Cache Misses']
sizes = [cache.hits, cache.misses]
colors = ['#2ecc71', '#e74c3c']
plt.pie(sizes, labels=labels, autopct='%1.1f%%', colors=colors, startangle=90)
plt.title(f'Cache Performance (Hit Rate: {cache.hit_rate():.1f}%)')
plt.show()

print(f"\n💡 Position caching can give 2-5x speedup in MCTS!")

### 2. Parallelization

**Goal**: Run multiple MCTS searches in parallel

**Speedup**: Near-linear with CPU cores (4x on 4 cores)

In [ ]:
import multiprocessing as mp
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor

print("MCTS Parallelization Demo\n")
print("="*70)

num_cores = mp.cpu_count()
print(f"Available CPU cores: {num_cores}")

def single_mcts_search(state, num_simulations):
    """Simulate one MCTS search."""
    time.sleep(0.01 * num_simulations)  # Simulate work
    return np.random.rand(81)  # Simulated policy

# Benchmark sequential
print("\n1. Sequential MCTS (baseline)")
state = env.reset()
num_searches = 8
num_sims = 100

start = time.time()
results_seq = []
for _ in range(num_searches):
    result = single_mcts_search(state, num_sims)
    results_seq.append(result)
time_sequential = time.time() - start

print(f"   Time: {time_sequential:.2f}s")
print(f"   Searches/sec: {num_searches/time_sequential:.1f}")

# Benchmark parallel (threads)
print("\n2. Parallel MCTS (threads)")
start = time.time()
with ThreadPoolExecutor(max_workers=4) as executor:
    results_thread = list(executor.map(
        lambda _: single_mcts_search(state, num_sims),
        range(num_searches)
    ))
time_thread = time.time() - start

print(f"   Time: {time_thread:.2f}s")
print(f"   Searches/sec: {num_searches/time_thread:.1f}")
print(f"   Speedup: {time_sequential/time_thread:.2f}x")

# Benchmark parallel (processes)
print("\n3. Parallel MCTS (processes)")
start = time.time()
with ProcessPoolExecutor(max_workers=4) as executor:
    results_process = list(executor.map(
        lambda _: single_mcts_search(state, num_sims),
        range(num_searches)
    ))
time_process = time.time() - start

print(f"   Time: {time_process:.2f}s")
print(f"   Searches/sec: {num_searches/time_process:.1f}")
print(f"   Speedup: {time_sequential/time_process:.2f}x")

# Visualize
fig, ax = plt.subplots(figsize=(10, 6))

methods = ['Sequential', 'Parallel\n(Threads)', 'Parallel\n(Processes)']
times = [time_sequential, time_thread, time_process]
speedups = [1.0, time_sequential/time_thread, time_sequential/time_process]

x = np.arange(len(methods))
width = 0.35

ax.bar(x - width/2, times, width, label='Time (s)', color='#3498db')
ax2 = ax.twinx()
ax2.bar(x + width/2, speedups, width, label='Speedup', color='#2ecc71')

ax.set_ylabel('Time (s)')
ax2.set_ylabel('Speedup (x)')
ax.set_title('MCTS Parallelization Performance')
ax.set_xticks(x)
ax.set_xticklabels(methods)
ax.legend(loc='upper left')
ax2.legend(loc='upper right')

plt.tight_layout()
plt.show()

print(f"\n💡 Parallel MCTS can give near-linear speedup with CPU cores!")

### 3. Early Termination

**Goal**: Stop MCTS when one move is clearly best

**Speedup**: 1.5-2x in positions with obvious moves

In [ ]:
print("MCTS Early Termination Demo\n")
print("="*70)

def early_termination_check(visit_counts, threshold=0.9):
    """Check if one move dominates.
    
    Args:
        visit_counts: Array of visit counts for each move
        threshold: Fraction of total visits for early stop
    
    Returns:
        True if should terminate early
    """
    if len(visit_counts) == 0:
        return False
    
    total_visits = np.sum(visit_counts)
    if total_visits == 0:
        return False
    
    max_visits = np.max(visit_counts)
    return (max_visits / total_visits) >= threshold

# Simulate MCTS with early termination
print("Simulating MCTS with early termination...\n")

max_simulations = 400
early_term_threshold = 0.8

# Scenario 1: Obvious move (early termination)
print("Scenario 1: Obvious move")
visit_counts_obvious = np.zeros(81)
best_move = 40

for sim in range(max_simulations):
    # Simulate: 80% visits to best move, 20% to others
    if np.random.rand() < 0.8:
        visit_counts_obvious[best_move] += 1
    else:
        visit_counts_obvious[np.random.randint(81)] += 1
    
    # Check early termination
    if early_termination_check(visit_counts_obvious, early_term_threshold):
        print(f"  ✓ Early termination at simulation {sim+1}/{max_simulations}")
        print(f"  Simulations saved: {max_simulations - sim - 1}")
        print(f"  Speedup: {max_simulations/(sim+1):.2f}x")
        break

# Scenario 2: Unclear position (no early termination)
print("\nScenario 2: Unclear position")
visit_counts_unclear = np.zeros(81)

for sim in range(max_simulations):
    # Simulate: Uniform distribution (no clear best)
    visit_counts_unclear[np.random.randint(81)] += 1
    
    # Check early termination
    if early_termination_check(visit_counts_unclear, early_term_threshold):
        print(f"  ✓ Early termination at simulation {sim+1}/{max_simulations}")
        break
else:
    print(f"  ✗ No early termination - ran all {max_simulations} simulations")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scenario 1
top_moves_1 = np.argsort(visit_counts_obvious)[-10:]
axes[0].barh(range(10), visit_counts_obvious[top_moves_1], color='#2ecc71')
axes[0].set_xlabel('Visit Count')
axes[0].set_title('Scenario 1: Obvious Move (Early Termination)')
axes[0].set_yticks(range(10))
axes[0].set_yticklabels([f'Move {m}' for m in top_moves_1])

# Scenario 2
top_moves_2 = np.argsort(visit_counts_unclear)[-10:]
axes[1].barh(range(10), visit_counts_unclear[top_moves_2], color='#e74c3c')
axes[1].set_xlabel('Visit Count')
axes[1].set_title('Scenario 2: Unclear Position (Full Search)')
axes[1].set_yticks(range(10))
axes[1].set_yticklabels([f'Move {m}' for m in top_moves_2])

plt.tight_layout()
plt.show()

print("\n💡 Early termination saves 30-50% of simulations in obvious positions!")

---

## Part 5: Training Optimization

### 1. Mixed Precision Training

**Goal**: Use FP16 instead of FP32 for faster training

**Speedup**: 2-3x on modern GPUs (with Tensor Cores)

In [ ]:
print("Mixed Precision Training Demo\n")
print("="*70)

# Check GPU capability
gpus = tf.config.list_physical_devices('GPU')
if len(gpus) > 0:
    print(f"GPU available: {gpus[0]}")
    
    # Enable mixed precision
    from tensorflow.keras import mixed_precision
    policy = mixed_precision.Policy('mixed_float16')
    mixed_precision.set_global_policy(policy)
    
    print(f"\n✓ Mixed precision enabled")
    print(f"  Compute dtype: {policy.compute_dtype}")
    print(f"  Variable dtype: {policy.variable_dtype}")
    print(f"\nExpected speedup: 2-3x on GPUs with Tensor Cores")
    print(f"Expected memory reduction: ~50%")
else:
    print("No GPU available - mixed precision not recommended for CPU")

print("\n💡 Mixed precision is essential for modern GPU training!")
print("\nExample usage:")
print("""
from tensorflow.keras import mixed_precision
policy = mixed_precision.Policy('mixed_float16')
mixed_precision.set_global_policy(policy)

# Build model as usual - will use FP16 automatically
agent = ModelBuilder().go(9).prometheus().build()

# Train 2-3x faster!
""")

### 2. Batch Size Tuning

**Goal**: Find optimal batch size for throughput

**Trade-off**: Larger batch = faster, but may not fit in memory

In [ ]:
print("Batch Size Tuning Demo\n")
print("="*70)

# Test different batch sizes
batch_sizes = [16, 32, 64, 128, 256]
throughputs = []

for batch_size in batch_sizes:
    print(f"\nTesting batch size {batch_size}...")
    
    # Create dummy data
    dummy_states = np.random.rand(batch_size, 9, 9, 3).astype(np.float32)
    dummy_policy = np.random.rand(batch_size, 81).astype(np.float32)
    dummy_value = np.random.rand(batch_size, 1).astype(np.float32)
    
    # Time training step
    try:
        times = []
        for _ in range(10):
            start = time.time()
            _ = agent.model.train_on_batch(
                dummy_states,
                {'policy': dummy_policy, 'value': dummy_value}
            )
            times.append(time.time() - start)
        
        mean_time = np.mean(times)
        throughput = batch_size / mean_time  # Samples/sec
        throughputs.append(throughput)
        
        print(f"  Time per batch: {mean_time*1000:.2f}ms")
        print(f"  Throughput: {throughput:.1f} samples/sec")
        
    except Exception as e:
        print(f"  ✗ Failed (OOM?): {e}")
        throughputs.append(0)
        break

# Find optimal
if len(throughputs) > 0 and max(throughputs) > 0:
    optimal_idx = np.argmax(throughputs)
    optimal_batch = batch_sizes[optimal_idx]
    optimal_throughput = throughputs[optimal_idx]
    
    print(f"\n" + "="*70)
    print(f"Optimal batch size: {optimal_batch}")
    print(f"Peak throughput: {optimal_throughput:.1f} samples/sec")
    print("="*70)
    
    # Visualize
    plt.figure(figsize=(10, 6))
    valid_batches = batch_sizes[:len(throughputs)]
    plt.plot(valid_batches, throughputs, marker='o', linewidth=2, markersize=10)
    plt.axvline(optimal_batch, color='red', linestyle='--', 
                label=f'Optimal: {optimal_batch}')
    plt.xlabel('Batch Size')
    plt.ylabel('Throughput (samples/sec)')
    plt.title('Training Throughput vs Batch Size')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()
    
    print("\n💡 Use largest batch size that fits in memory for best throughput!")

---

## Part 6: Summary and Best Practices

### Optimization Summary

| Technique | Speedup | Difficulty | When to Use |
|-----------|---------|------------|-------------|
| **Model Quantization** | 2-4x | Easy | Always (deployment) |
| **Position Caching** | 2-5x | Easy | Always (MCTS) |
| **Parallelization** | 2-8x | Medium | Multi-core systems |
| **Mixed Precision** | 2-3x | Easy | Modern GPUs |
| **Batch Size Tuning** | 1.5-2x | Easy | Always (training) |
| **Model Pruning** | 1.5-2x | Hard | Advanced users |
| **Knowledge Distillation** | 3-5x | Hard | Small models needed |
| **Early Termination** | 1.5-2x | Easy | MCTS |

### Quick Wins (Easiest, Highest Impact)

1. **Enable position caching** in MCTS (5 min, 2-5x speedup)
2. **Quantize deployed models** (10 min, 2-4x speedup)
3. **Use mixed precision** on GPU (2 min, 2-3x speedup)
4. **Tune batch size** (15 min, 1.5-2x speedup)
5. **Add early termination** to MCTS (10 min, 1.5-2x speedup)

**Total potential speedup: 48-240x** (compound effect)

### Optimization Checklist

**Training**:
- [ ] Profile code to find bottlenecks
- [ ] Enable mixed precision (GPU only)
- [ ] Tune batch size for throughput
- [ ] Use data pipeline optimization (prefetch, cache)
- [ ] Consider distributed training for large-scale

**MCTS**:
- [ ] Implement position caching
- [ ] Enable parallelization (multi-core)
- [ ] Add early termination
- [ ] Optimize simulation count based on time

**Deployment**:
- [ ] Quantize model to INT8
- [ ] Use TensorFlow Lite or ONNX
- [ ] Enable batching for multiple requests
- [ ] Consider TensorRT for NVIDIA GPUs
- [ ] Monitor inference latency

### Performance Targets

**9×9 Go**:
- Inference: <10ms (excellent), <50ms (good)
- MCTS 400 sims: <1s (excellent), <2s (good)
- Training: >100 games/min (excellent), >50 games/min (good)

**19×19 Go**:
- Inference: <50ms (excellent), <200ms (good)
- MCTS 400 sims: <2s (excellent), <5s (good)
- Training: >20 games/min (excellent), >10 games/min (good)

### Next Steps

1. **Profile your code**: Find the actual bottlenecks
2. **Apply quick wins**: Start with easiest optimizations
3. **Measure improvements**: Benchmark before and after
4. **Iterate**: Continue optimizing highest-impact areas
5. **Document**: Track what works for your specific use case

---

**Congratulations!** You now know how to optimize Prometheus for maximum performance! 🚀

**Remember**: 
- ⚡ Profile before optimizing
- 🎯 Focus on bottlenecks
- 📊 Measure improvements
- 🔁 Iterate continuously